# 🪟 Day 04a: Window Functions — ROW_NUMBER, RANK, LAG, LEAD, PARTITION BY

---

## 🎯 What You'll Master Today
- ROW_NUMBER, RANK, DENSE_RANK
- LAG, LEAD (previous/next row access)
- SUM/AVG/COUNT OVER (running aggregates)
- PARTITION BY vs GROUP BY
- ROWS BETWEEN (window frames)

---

## 🏗️ The Sliding Window Analogy

> GROUP BY collapses rows into one per group (you lose individual rows).
> Window functions keep every row AND add group-level info beside each one —
> like sliding a magnifying glass over a spreadsheet while seeing the full sheet.

```
╔══════════════════════════════════════════════════════════════════╗
║            GROUP BY vs WINDOW FUNCTION                          ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  GROUP BY:                                                       ║
║  Engineering → 1 row (avg=92,500)                               ║
║  Marketing   → 1 row (avg=70,000)                               ║
║                                                                  ║
║  WINDOW:                                                         ║
║  Alice   | Engineering | 95000 | dept_avg=92500                  ║
║  Bob     | Engineering | 85000 | dept_avg=92500                  ║
║  Eve     | Engineering | 92000 | dept_avg=92500                  ║
║  Hank    | Engineering | 98000 | dept_avg=92500                  ║
║  Charlie | Marketing   | 72000 | dept_avg=70000                  ║
║  Diana   | Marketing   | 68000 | dept_avg=70000                  ║
║                                                                  ║
║  → Every row preserved + aggregate column added!                ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# Setup
# ============================================

employees = [
    {"id": 1, "name": "Alice",   "dept": "Engineering", "salary": 95000},
    {"id": 2, "name": "Bob",     "dept": "Engineering", "salary": 85000},
    {"id": 3, "name": "Charlie", "dept": "Marketing",   "salary": 72000},
    {"id": 4, "name": "Diana",   "dept": "Marketing",   "salary": 68000},
    {"id": 5, "name": "Eve",     "dept": "Engineering", "salary": 92000},
    {"id": 6, "name": "Frank",   "dept": "Sales",       "salary": 65000},
    {"id": 7, "name": "Grace",   "dept": "Sales",       "salary": 71000},
    {"id": 8, "name": "Hank",    "dept": "Engineering", "salary": 98000},
    {"id": 9, "name": "Ivy",     "dept": "Marketing",   "salary": 72000},  # tie with Charlie!
]

def print_table(rows, columns=None):
    if not rows:
        print("  (empty)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {c: max(len(str(c)), max(len(str(r.get(c,''))) for r in rows)) for c in columns}
    header = "  " + " | ".join(f"{c:<{widths[c]}}" for c in columns)
    print(header)
    print("  " + "-+-".join("-"*widths[c] for c in columns))
    for r in rows:
        print("  " + " | ".join(f"{str(r.get(c,'')):<{widths[c]}}" for c in columns))

In [ ]:
# ============================================
# 1. ROW_NUMBER vs RANK vs DENSE_RANK
# ============================================

# SQL: SELECT name, dept, salary,
#        ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) as row_num,
#        RANK()       OVER (PARTITION BY dept ORDER BY salary DESC) as rnk,
#        DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) as dense_rnk
#      FROM employees;

from collections import defaultdict

print("  ROW_NUMBER vs RANK vs DENSE_RANK (PARTITION BY dept ORDER BY salary DESC)\n")

groups = defaultdict(list)
for e in employees:
    groups[e["dept"]].append(e)

result = []
for dept in sorted(groups):
    sorted_emps = sorted(groups[dept], key=lambda x: x["salary"], reverse=True)
    rank = 1
    dense = 1
    for i, e in enumerate(sorted_emps):
        if i > 0 and e["salary"] < sorted_emps[i-1]["salary"]:
            rank = i + 1
            dense += 1
        elif i > 0 and e["salary"] == sorted_emps[i-1]["salary"]:
            pass  # rank stays same, dense stays same
        result.append({"name": e["name"], "dept": dept, "salary": e["salary"],
                       "row_num": i + 1, "rank": rank, "dense_rank": dense})

print_table(result)
print("\n  💡 Charlie & Ivy tie at 72000 → RANK both 1, but ROW_NUMBER differs!")
print("  💡 RANK skips: 1,1,3. DENSE_RANK doesn't skip: 1,1,2")

In [ ]:
# ============================================
# 2. Top-N per group (classic interview Q!)
# ============================================

# SQL: WITH ranked AS (
#        SELECT *, ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS rn
#        FROM employees
#      )
#      SELECT * FROM ranked WHERE rn <= 2;

print("  Top 2 earners per department (ROW_NUMBER + filter)\n")

top_n = [r for r in result if r["row_num"] <= 2]
print_table(top_n, ["name", "dept", "salary", "row_num"])
print("\n  💡 This is THE most common window function interview question!")

In [ ]:
# ============================================
# 3. LAG and LEAD — access prev/next row
# ============================================

# SQL: SELECT name, salary,
#        LAG(salary, 1) OVER (ORDER BY salary) AS prev_salary,
#        LEAD(salary, 1) OVER (ORDER BY salary) AS next_salary,
#        salary - LAG(salary, 1) OVER (ORDER BY salary) AS diff
#      FROM employees ORDER BY salary;

print("  LAG/LEAD: salary differences between consecutive rows\n")

sorted_emps = sorted(employees, key=lambda x: x["salary"])
result = []
for i, e in enumerate(sorted_emps):
    prev_sal = sorted_emps[i-1]["salary"] if i > 0 else None
    next_sal = sorted_emps[i+1]["salary"] if i < len(sorted_emps)-1 else None
    result.append({
        "name": e["name"], "salary": e["salary"],
        "lag_salary": prev_sal if prev_sal else "NULL",
        "lead_salary": next_sal if next_sal else "NULL",
        "diff_from_prev": e["salary"] - prev_sal if prev_sal else "NULL",
    })

print_table(result)
print("\n  💡 LAG(col, N): N rows back. LEAD(col, N): N rows ahead. Default offset=1")

In [ ]:
# ============================================
# 4. Running Aggregates — SUM/AVG OVER
# ============================================

# SQL: SELECT name, salary,
#        SUM(salary) OVER (ORDER BY salary ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_sum,
#        AVG(salary) OVER (PARTITION BY dept) AS dept_avg
#      FROM employees;

print("  Running SUM + Department AVG\n")

dept_avgs = {}
for dept, emps in groups.items():
    dept_avgs[dept] = round(sum(e["salary"] for e in emps) / len(emps))

sorted_emps = sorted(employees, key=lambda x: x["salary"])
running = 0
result = []
for e in sorted_emps:
    running += e["salary"]
    result.append({
        "name": e["name"], "salary": e["salary"],
        "running_sum": running,
        "dept_avg": dept_avgs[e["dept"]],
    })

print_table(result)
print("\n  💡 PARTITION BY = 'group within window'. ORDER BY = 'running within partition'")

In [ ]:
# ============================================
# 5. Window Frame Cheat Sheet
# ============================================

print("""
  ╔════════════════════════════════════════════════════════════════════╗
  ║                  WINDOW FRAME SYNTAX                              ║
  ╠════════════════════════════════════════════════════════════════════╣
  ║                                                                    ║
  ║  func() OVER (                                                     ║
  ║    PARTITION BY col         -- split into groups                    ║
  ║    ORDER BY col             -- order within partition               ║
  ║    ROWS BETWEEN ... AND ... -- frame boundaries                    ║
  ║  )                                                                 ║
  ║                                                                    ║
  ╠════════════════════════════════════════════════════════════════════╣
  ║  Frame Option                │ Meaning                            ║
  ╠══════════════════════════════╪════════════════════════════════════╣
  ║  UNBOUNDED PRECEDING         │ From partition start               ║
  ║  N PRECEDING                 │ N rows before current              ║
  ║  CURRENT ROW                 │ Current row                        ║
  ║  N FOLLOWING                 │ N rows after current               ║
  ║  UNBOUNDED FOLLOWING         │ To partition end                   ║
  ╠══════════════════════════════╧════════════════════════════════════╣
  ║                                                                    ║
  ║  3-row moving average:                                             ║
  ║  AVG(salary) OVER (ORDER BY date ROWS BETWEEN 1 PRECEDING         ║
  ║                    AND 1 FOLLOWING)                                 ║
  ║                                                                    ║
  ║  Running total:                                                    ║
  ║  SUM(amt) OVER (ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING    ║
  ║                 AND CURRENT ROW)                                    ║
  ║                                                                    ║
  ╚════════════════════════════════════════════════════════════════════╝
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | ROW_NUMBER vs RANK vs DENSE_RANK? | ROW_NUMBER: unique. RANK: ties skip. DENSE_RANK: ties don't skip |
| 2 | Top-N per group? | ROW_NUMBER() OVER (PARTITION BY grp ORDER BY col DESC) + WHERE rn <= N |
| 3 | PARTITION BY vs GROUP BY? | GROUP BY collapses rows. PARTITION BY keeps all rows + adds aggregate |
| 4 | Running total in SQL? | SUM(col) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING) |
| 5 | What does ROWS BETWEEN do? | Defines the window frame (which rows the function operates on) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Window = aggregate WITHOUT collapsing rows            │
## │  • Top-N per group: ROW_NUMBER + PARTITION BY + filter   │
## │  • LAG/LEAD: access previous/next row values             │
## │  • ROWS BETWEEN: controls which rows feed the function   │
## │  • RANK skips gaps on ties, DENSE_RANK doesn't           │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **SQL Practice** — 15+ interview-grade SQL problems